# Conjugation boxes: Pauli gadgets and two-term LCU

**Download Notebook** - {nb-download}`conjugation_box.ipynb`

Build an XYZ Pauli gadget and a two-term LCU, then apply call, dagger, control,
and controlled dagger through `ConjugationBox`.

Run cells in order. Install dependencies with `uv sync --extra dev-dependencies`.

In [1]:
import numpy as np
import zixy.qubit.pauli as zqp
from scipy.linalg import expm

from guppylang import guppy
from guppylang.std.angles import angle
from guppylang.std.builtins import array, control, dagger
from guppylang.std.quantum import cx, cz, h, qubit, ry, rz, sdg, x

from guppyalgos.primitives.subroutines.conjugation import ConjugationBox
from guppyalgos.testing import (
    align_phase, assert_unitary_modifiers, get_unitary, get_unitary_projected,
)

# Guppy uses half-turns; alpha is the angle in radians.
theta = 0.37
alpha = np.pi * theta

## 1. What does the box do?

G must be daggerable and U unitary. Both act on the same register.
`box.compose(qs)` executes **G → U → G†**; matrices act right to left:

$$
W=G^{\dagger}UG,
\qquad
W^{\dagger}=G^{\dagger}U^{\dagger}G.
$$

| Call context | Execution order |
| --- | --- |
| Ordinary call | G → U → G† |
| `with dagger:` | G → U† → G† |
| `with control(controls):` | G → controlled-U → G† |
| `with control(controls), dagger:` | G → controlled-U† → G† |

Only U receives the external control c:

$$
\mathrm{C}(W)=(I_c\otimes G^{\dagger})\,\mathrm{C}(U)\,(I_c\otimes G).
$$

- c = 0: U is skipped and G† G cancels.
- c = 1: the box applies W.

Use U† in place of U for the controlled adjoint.

### How the checks work

Each modifier below has its own function. We compare its simulated matrix with
the reference, up to global phase, and assert unitarity. Controlled functions
are projected onto both control branches: c = 0 gives identity and c = 1 gives
the operation. Successful assertions return `True`.

`assert_unitary_modifiers` additionally checks the full controlled matrices,
including the relative phase between the two branches.

In [2]:
def check_unitary(actual, expected):
    unitary = np.allclose(actual.conj().T @ actual, np.eye(actual.shape[0]), atol=1e-8)
    matches = np.allclose(align_phase(actual), align_phase(expected), atol=1e-8)
    passed = bool(unitary and matches)
    assert passed
    return passed


def check_controlled_blocks(circuit, width, expected):
    results = {}
    for active in (False, True):
        projection = {"control": [active]}
        block = get_unitary_projected(
            circuit,
            n_state_qubits=width,
            post_select_dict=projection,
            pre_select_dict=projection,
        )
        reference = expected if active else np.eye(2**width)
        results[f"control={int(active)}"] = check_unitary(block, reference)
    return results

## 2. An XYZ Pauli gadget

For $P=X_0Y_1Z_2$, G applies H on q0, S† then H on q1, and two parity CNOTs.
This gives $G^{\dagger}Z_2G=P$. With $U=R_z(\alpha)$ on q2 and
$\alpha=\pi\theta$:

$$
W_{XYZ}(\theta)=G^{\dagger}R_z(\alpha)G=e^{-i\alpha X_0Y_1Z_2/2}.
$$

In [3]:
@guppy(daggerable=True)
def g(qs: array[qubit, 3]) -> None:
    h(qs[0])
    sdg(qs[1])
    h(qs[1])
    cx(qs[0], qs[2])
    cx(qs[1], qs[2])


@guppy(unitary=True)
def u(qs: array[qubit, 3]) -> None:
    rz(qs[2], angle(theta))

### One function for each modifier

Only the middle $\mathrm{C}R_z(\alpha)$ receives the external control;
the basis changes and parity CNOTs remain uncontrolled.

In [4]:
@guppy(unitary=True)
def pauli_gadget(qs: array[qubit, 3]) -> None:
    box = ConjugationBox(g, u)
    box.compose(qs)


@guppy
def pauli_daggered(qs: array[qubit, 3]) -> None:
    box = ConjugationBox(g, u)
    with dagger:
        box.compose(qs)


@guppy
def pauli_controlled(controls: array[qubit, 1], qs: array[qubit, 3]) -> None:
    box = ConjugationBox(g, u)
    with control(controls):
        box.compose(qs)


@guppy
def pauli_ctrl_daggered(controls: array[qubit, 1], qs: array[qubit, 3]) -> None:
    box = ConjugationBox(g, u)
    with control(controls), dagger:
        box.compose(qs)

### Simulate and check

Build $W=e^{-i\alpha X_0Y_1Z_2/2}$ with Zixy in big-endian order. The dagger
uses $W^{\dagger}$; each controlled function is checked with the control
prepared and projected onto the same value.

In [5]:
pauli_hamiltonian = zqp.RealTermSum.from_str("(1.0, X0 Y1 Z2)", 3)
pauli_matrix = pauli_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
pauli_reference = expm(-0.5j * alpha * pauli_matrix)

pauli_checks = {
    "call": check_unitary(get_unitary(pauli_gadget, 3), pauli_reference),
    "daggered": check_unitary(
        get_unitary(pauli_daggered, 3), pauli_reference.conj().T
    ),
    "controlled": check_controlled_blocks(pauli_controlled, 3, pauli_reference),
    "ctrl_daggered": check_controlled_blocks(
        pauli_ctrl_daggered, 3, pauli_reference.conj().T
    ),
}
assert_unitary_modifiers(pauli_gadget, 3, expected_unitary=pauli_reference)
pauli_checks

{'call': True,
 'daggered': True,
 'controlled': {'control=0': True, 'control=1': True},
 'ctrl_daggered': {'control=0': True, 'control=1': True}}

## 3. A two-term LCU block encoding

Use `qs[0]` as index a and `qs[1]` as target t. G = PREPARE rotates a:

$$
R_y(\alpha)|0\rangle=c_0|0\rangle+c_1|1\rangle,
\qquad
c_0=\cos(\alpha/2),\quad c_1=\sin(\alpha/2).
$$

U = SELECT applies X for index zero and Z for index one:

$$
\mathrm{SELECT}=|0\rangle\langle0|_a\otimes X_t
+|1\rangle\langle1|_a\otimes Z_t.
$$

The box implements

$$
B=\mathrm{PREPARE}^{\dagger}\,\mathrm{SELECT}\,\mathrm{PREPARE},
$$

with index-zero block

$$
(\langle0|_a\otimes I_t)\,B\,(|0\rangle_a\otimes I_t)
=c_0^2X_t+c_1^2Z_t=A.
$$

B is unitary; A need not be. Keep the index qubit when checking B's modifiers.

In [6]:
@guppy(daggerable=True)
def prepare(qs: array[qubit, 2]) -> None:
    ry(qs[0], angle(theta))


@guppy(unitary=True)
def select(qs: array[qubit, 2]) -> None:
    # Open-index control: apply X to the target when the index is zero.
    x(qs[0])
    cx(qs[0], qs[1])
    x(qs[0])
    # Closed-index control: apply Z to the target when the index is one.
    cz(qs[0], qs[1])

### Distinguish the index from the external control

Index a chooses X or Z. External control c turns the whole SELECT block on or off.

- c = 0: SELECT is skipped and PREPARE† PREPARE cancels.
- c = 1: SELECT uses a to choose the Pauli.
- PREPARE and PREPARE† always run.

The control covers SELECT's open-index X gates too. It controls the full block
encoding B, including its index qubit, rather than A alone.

In [7]:
@guppy(unitary=True)
def lcu(qs: array[qubit, 2]) -> None:
    box = ConjugationBox(prepare, select)
    box.compose(qs)


@guppy
def lcu_daggered(qs: array[qubit, 2]) -> None:
    box = ConjugationBox(prepare, select)
    with dagger:
        box.compose(qs)


@guppy
def lcu_controlled(controls: array[qubit, 1], qs: array[qubit, 2]) -> None:
    box = ConjugationBox(prepare, select)
    with control(controls):
        box.compose(qs)


@guppy
def lcu_ctrl_daggered(controls: array[qubit, 1], qs: array[qubit, 2]) -> None:
    box = ConjugationBox(prepare, select)
    with control(controls), dagger:
        box.compose(qs)

### Check the four unitary modes

Construct PREPARE and SELECT with Zixy, using

$$
\mathrm{SELECT}=\tfrac12(X_t+Z_aX_t+Z_t-Z_aZ_t),
\qquad B=\mathrm{PREPARE}^{\dagger}\,\mathrm{SELECT}\,\mathrm{PREPARE}.
$$

For the controlled modes, project the external control only. The index qubit
stays part of B, so these projected blocks are still unitary.

In [8]:
prepare_hamiltonian = zqp.RealTermSum.from_str("(1.0, Y0)", 2)
prepare_matrix = expm(
    -0.5j * alpha * prepare_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
)
select_hamiltonian = zqp.RealTermSum.from_str(
    "(0.5, X1), (0.5, Z0 X1), (0.5, Z1), (-0.5, Z0 Z1)", 2
)
select_matrix = select_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
lcu_reference = prepare_matrix.conj().T @ select_matrix @ prepare_matrix
lcu_unitary = get_unitary(lcu, 2)

lcu_checks = {
    "call": check_unitary(lcu_unitary, lcu_reference),
    "daggered": check_unitary(get_unitary(lcu_daggered, 2), lcu_reference.conj().T),
    "controlled": check_controlled_blocks(lcu_controlled, 2, lcu_reference),
    "ctrl_daggered": check_controlled_blocks(
        lcu_ctrl_daggered, 2, lcu_reference.conj().T
    ),
}
assert_unitary_modifiers(lcu, 2, expected_unitary=lcu_reference)
lcu_checks

{'call': True,
 'daggered': True,
 'controlled': {'control=0': True, 'control=1': True},
 'ctrl_daggered': {'control=0': True, 'control=1': True}}

### Project the LCU index onto zero

Now project the index input and output, giving $A=c_0^2X+c_1^2Z$.
In big-endian order (index, target), this is the top-left block of the simulated
B. A need not be unitary; compare its matrix without renormalizing it.

In [9]:
weight_x = np.cos(alpha / 2) ** 2
weight_z = np.sin(alpha / 2) ** 2
lcu_hamiltonian = zqp.RealTermSum.from_str(
    f"({weight_x}, X0), ({weight_z}, Z0)", 1
)
expected_block = lcu_hamiltonian.to_sparse_matrix(big_endian=True).toarray()
lcu_projected = lcu_unitary[:2, :2]
projection_matches = np.allclose(
    align_phase(lcu_projected), align_phase(expected_block), atol=1e-8
)
assert projection_matches
bool(projection_matches)

True